In [ ]:
# Import python packages
import streamlit as st
import pandas as pd

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
-- Describe the MD of Movie table
DESC TABLE QUICKSIGHTSA.MOVIES.MOVIES_CURATED;

In [ ]:
-- View sample data of Movie table
Select * from QUICKSIGHTSA.MOVIES.MOVIES_CURATED limit 10;

In [ ]:
-- Describe the MD of Rating table
DESC TABLE QUICKSIGHTSA.MOVIES.RATINGS_CURATED ;

In [ ]:
-- Sample data of rating table
Select * from QUICKSIGHTSA.MOVIES.RATINGS_CURATED limit 10;

In [ ]:
-- Desribe the MD of user table
DESC TABLE QUICKSIGHTSA.MOVIES.users_curated ;

In [ ]:
-- Sample data of user table
select * from QUICKSIGHTSA.MOVIES.users_curated limit 10;

In [ ]:
-- Define the Semantic View
CREATE OR REPLACE SEMANTIC VIEW MOVIE_ANALYTICS_SV
TABLES (
  MOVIES AS QUICKSIGHTSA.MOVIES.MOVIES_CURATED
    PRIMARY KEY (MOVIEID)
    WITH SYNONYMS ('movies', 'films')
    COMMENT = 'Movie dimension',

  USERS AS QUICKSIGHTSA.MOVIES.USERS_CURATED
    PRIMARY KEY (USERID)
    WITH SYNONYMS ('users', 'audience', 'viewers')
    COMMENT = 'User dimension',

  RATINGS AS QUICKSIGHTSA.MOVIES.RATINGS_CURATED
    PRIMARY KEY (USERID, MOVIEID)
    WITH SYNONYMS ('ratings', 'reviews')
    COMMENT = 'Movie ratings fact'
)

RELATIONSHIPS (
  RATINGS_TO_MOVIES AS
    RATINGS(MOVIEID) REFERENCES MOVIES,

  RATINGS_TO_USERS AS
    RATINGS(USERID) REFERENCES USERS
)

FACTS (
  RATINGS.rating_value AS RATINGS.RATING
)

DIMENSIONS (
  -- Movie dimensions
  MOVIES.movie_title AS MOVIES.TITLE
    WITH SYNONYMS = ('film title', 'movie name')
    COMMENT = 'Title of the movie',
  MOVIES.release_year AS MOVIES.RELEASE
    COMMENT = 'Year the movie was released',

  -- User dimensions
  USERS.user_full_name AS CONCAT(USERS.FIRSTNAME, ' ', USERS.LASTNAME)
    WITH SYNONYMS = ('audience name', 'viewer name')
    COMMENT = 'Full name of the user',
  USERS.user_first_name AS USERS.FIRSTNAME
    COMMENT = 'First name of the user',
  USERS.user_email AS USERS.EMAIL
    WITH SYNONYMS = ('email address', 'contact email')
    COMMENT = 'Email address of the user',
  USERS.user_last_name AS USERS.LASTNAME
    COMMENT = 'Last name of the user',
  USERS.user_city AS USERS.CITY
    COMMENT = 'City where user is located',
  USERS.user_state AS USERS.STATE
    COMMENT = 'State where user is located',
  USERS.user_country AS USERS.COUNTRY
    COMMENT = 'Country where user is located',

  -- ID dimensions
  USERS.user_id AS USERS.USERID
    COMMENT = 'User ID',
  MOVIES.movie_id AS MOVIES.MOVIEID
    COMMENT = 'Movie ID',

  -- Rating timestamp dimensions
  RATINGS.rating_timestamp AS RATINGS.TIMESTAMP
    WITH SYNONYMS = ('review date', 'rating date')
    COMMENT = 'When the rating was given',
  RATINGS.rating_year AS YEAR(RATINGS.TIMESTAMP)
    WITH SYNONYMS = ('review year', 'year rated')
    COMMENT = 'Year when the rating was given',
  RATINGS.rating_month AS MONTH(RATINGS.TIMESTAMP)
    WITH SYNONYMS = ('review month', 'month rated')
    COMMENT = 'Month when the rating was given',
  RATINGS.rating_day AS DAY(RATINGS.TIMESTAMP)
    WITH SYNONYMS = ('review day', 'day rated')
    COMMENT = 'Day when the rating was given',

  -- Rating ID dimensions
  RATINGS.user_id AS RATINGS.USERID
    COMMENT = 'User ID from ratings table',
  RATINGS.movie_id AS RATINGS.MOVIEID
    COMMENT = 'Movie ID from ratings table'
)

METRICS (
  RATINGS.total_ratings AS COUNT(RATINGS.rating_value)
    WITH SYNONYMS = ('total reviews', 'rating count', 'number of ratings')
    COMMENT = 'Count of all rating values in the ratings table',

  RATINGS.avg_rating AS AVG(RATINGS.rating_value)
    WITH SYNONYMS = ('average rating', 'mean score', 'rating average')
    COMMENT = 'Average of all rating values in the ratings table',

  USERS.distinct_users AS COUNT(DISTINCT USERS.user_id)
    WITH SYNONYMS = ('total users', 'user count', 'number of users')
    COMMENT = 'Count of distinct user IDs from the users table',

  MOVIES.distinct_movies AS COUNT(DISTINCT MOVIES.movie_id)
    WITH SYNONYMS = ('total movies', 'movie count', 'number of movies')
    COMMENT = 'Count of distinct movie IDs from the movies table',

  RATINGS.distinct_users AS COUNT(DISTINCT RATINGS.user_id)
    WITH SYNONYMS = ('active users', 'users who rated', 'rating participants')
    COMMENT = 'Count of distinct user IDs from the ratings table',

  RATINGS.distinct_movies AS COUNT(DISTINCT RATINGS.movie_id)
    WITH SYNONYMS = ('rated movies', 'movies with ratings', 'reviewed films')
    COMMENT = 'Count of distinct movie IDs from the ratings table',

  RATINGS.popularity_score AS RATINGS.total_ratings * RATINGS.avg_rating
    WITH SYNONYMS = ('movie popularity', 'engagement score', 'quality index')
    COMMENT = 'Popularity score combining volume and quality'
)

--FILTER (
 -- recent_ratings AS RATINGS.rating_timestamp >= DATEADD(year, -1, CURRENT_TIMESTAMP())
 --   COMMENT = 'Ratings submitted in the last year',

 -- high_ratings AS RATINGS.rating_value >= 4
 --   COMMENT = 'Ratings with score 4 or higher'
--)

COMMENT = 'Semantic view for movie ratings and user behavior analytics, including time-grain dimensions and popularity score';

In [ ]:
SELECT * FROM SEMANTIC_VIEW (
  MOVIE_ANALYTICS_SV
  DIMENSIONS RATINGS.rating_year
  METRICS RATINGS.total_ratings, RATINGS.avg_rating
);

In [ ]:
SELECT * FROM SEMANTIC_VIEW (
  MOVIE_ANALYTICS_SV
  DIMENSIONS MOVIES.movie_title
  METRICS RATINGS.total_ratings, RATINGS.avg_rating
)
ORDER BY total_ratings DESC
LIMIT 10;

In [ ]:
-- 6. Top movies by release year
SELECT * FROM SEMANTIC_VIEW (
  MOVIE_ANALYTICS_SV
  DIMENSIONS MOVIES.release_year, MOVIES.movie_title
  METRICS RATINGS.total_ratings, RATINGS.avg_rating
)
WHERE release_year >= 2020
ORDER BY total_ratings DESC;

In [ ]:
-- 13. Highly rated movies released after 2015
SELECT * FROM SEMANTIC_VIEW (
  MOVIE_ANALYTICS_SV
  DIMENSIONS MOVIES.movie_title, MOVIES.release_year
  METRICS RATINGS.avg_rating, RATINGS.total_ratings
)
WHERE release_year > 2015 
  AND avg_rating >= 4.0
ORDER BY avg_rating DESC;

In [ ]:
-- 11. Countries with the highest average ratings
SELECT * FROM SEMANTIC_VIEW (
  MOVIE_ANALYTICS_SV
  DIMENSIONS USERS.user_country
  METRICS RATINGS.avg_rating, RATINGS.total_ratings
)
ORDER BY avg_rating DESC;

In [ ]:
-- 9. Top 10 movies by popularity score
SELECT * FROM SEMANTIC_VIEW (
  MOVIE_ANALYTICS_SV
  DIMENSIONS MOVIES.movie_title
  METRICS RATINGS.total_ratings, RATINGS.avg_rating
)
ORDER BY total_ratings DESC
LIMIT 10;

In [ ]:
import snowflake.connector
import boto3
import json
import os

In [ ]:
def get_secret(secret_name, region_name='us-east-1'):
    """Retrieve secret from AWS Secrets Manager"""
    client = boto3.client('secretsmanager', region_name=region_name)
    response = client.get_secret_value(SecretId=secret_name)
    return json.loads(response['SecretString'])

# Configuration
SNOWFLAKE_SECRET_NAME = os.getenv('SNOWFLAKE_SECRET_NAME', 'snowflake-credentials')
AWS_SECRET_NAME = os.getenv('AWS_SECRET_NAME', 'aws-credentials')
S3_BUCKET = os.getenv('S3_BUCKET', 'my-snowflake-extract')

SNOWFLAKE_CONFIG = get_secret(SNOWFLAKE_SECRET_NAME)
AWS_CONFIG = get_secret(AWS_SECRET_NAME)

In [ ]:
def describe_semantic_view(semantic_view_name):
    """Get semantic view metadata"""
    conn = snowflake.connector.connect(
        user=SNOWFLAKE_CONFIG['user'],
        password=SNOWFLAKE_CONFIG['pat_token'],
        account=SNOWFLAKE_CONFIG['account'],
        warehouse=SNOWFLAKE_CONFIG['warehouse'],
        database=SNOWFLAKE_CONFIG['database'],
        schema=SNOWFLAKE_CONFIG['schema'],
        insecure_mode=True
    )
    cursor = conn.cursor()
    
    try:
        query = f"DESCRIBE SEMANTIC VIEW {semantic_view_name}"
        cursor.execute(query)
        
        results = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description]
        metadata = [dict(zip(columns, row)) for row in results]
        
        return metadata
        
    finally:
        cursor.close()
        conn.close()

In [ ]:
# Then, we can use the python name to turn cell2 into a Pandas dataframe
my_df = cell2.to_pandas()

# Chart the data
st.subheader("Chance of SNOW ❄️")
st.line_chart(my_df, x='SNOWDAY', y='CHANCE_OF_SNOW')

# Give it a go!
st.subheader("Try it out yourself and show off your skills 🥇")